# Proyecto chatbot para Educacion

Se propone el desarrollo de una prueba de concepto de un chatbot educativo que utilice la metodología RAG (Retrieval Augmented Generation) para responder de manera precisa a preguntas sobre el contenido de un curso universitario. Este chatbot, alimentado con las notas de clase de un curso existente, funcionará como un tutor virtual personalizado, disponible las 24 horas del día para los estudiantes. La iniciativa tiene como objetivo principal mejorar la experiencia de aprendizaje al proporcionar un recurso adicional para consultar el conocimiento contenido en los apuntes elaborados por los docentes. El chatbot será capaz de comprender preguntas complejas, encontrar la información relevante en las notas de clase y generar respuestas coherentes y concisas. Se utilizarán redes neuronales preentrenadas, de acceso abierto y alojadas de forma local en el servidor de la facultad. Para el desarrollo de la prueba de concepto se utilizará el lenguaje Python. El proyecto abarca desde la recopilación y procesamiento de las notas de clase en formato PDF, Word u otro formato similar, hasta el desarrollo de una interfaz conversacional intuitiva mediante la librería Streamlit o similar.
Se espera que al utilizar el chatbot la consulta de los apuntes de clase por parte de los estudiantes aumente, de modo que se logre un aprendizaje más personalizado, una mayor comprensión de los conceptos y un ahorro de tiempo para los estudiantes. Además, se espera que este chatbot sea una herramienta valiosa para los docentes, al proporcionarles información sobre las áreas en las que los estudiantes tienen más interés o dificultades.

## Setup

In [1]:
import os
import numpy as np
#Retrieval libraries
from unidecode import unidecode
import re
import torch
from typing import List, Dict, Any
from langchain.schema import Document
from langchain.text_splitter import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings, LlamaCppEmbeddings

from langchain_community.vectorstores import Chroma
#from langchain.retrievers.document_compressors import FlashrankRerank
from langchain.retrievers import ContextualCompressionRetriever
#from langchain_community.document_compressors import SentenceTransformerRerank
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain.retrievers.document_compressors import EmbeddingsFilter # <-- Importar
from langchain.retrievers.document_compressors import DocumentCompressorPipeline
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

#LLM libraries

#from transformers import pipeline
#from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

In [2]:
import sys
from pathlib import Path

# Obtiene la ruta de la carpeta raíz (Edu-Bot) subiendo un nivel desde la notebook
root_dir = Path.cwd().parent

# Agrega la carpeta raíz a sys.path si no está incluida
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

In [9]:
#Actualizar archivos sin reiniciar kernel
%load_ext autoreload
%autoreload 2

## Setup Fase Generativa

In [13]:
from src.setup_key import get_api_key
import os 

try:
    my_llm_key = get_api_key()
        
        # Now you can use the key in your application
    print("Successfully loaded API Key.")
        # For security, we only show the first and last few characters
    print(f"Key starts with: {my_llm_key[:4]}... and ends with: ...{my_llm_key[-4:]}")
        
except ValueError as e:
    print(f"Error: {e}")
        
os.environ["GOOGLE_API_KEY"] = my_llm_key

Successfully loaded API Key.
Key starts with: AIza... and ends with: ...SZm4


In [22]:
from src.model_factory import ModelFactory

llm = ModelFactory.create_model(provider="llamacpp",n_ctx=256)

llama_model_loader: loaded meta data with 36 key-value pairs and 147 tensors from /home/nico/.cache/llama.cpp/unsloth_Llama-3.2-1B-Instruct-GGUF_Llama-3.2-1B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Llama-3.2-1B-Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Llama-3.2-1B-Instruct
llama_model_loader: - kv   5:                       general.quantized_by str              = Unsloth
llama_model_loader: - kv   6:                         general.size_labe

load: control token: 128146 '<|reserved_special_token_138|>' is not marked as EOG
load: control token: 128144 '<|reserved_special_token_136|>' is not marked as EOG
load: control token: 128142 '<|reserved_special_token_134|>' is not marked as EOG
load: control token: 128141 '<|reserved_special_token_133|>' is not marked as EOG
load: control token: 128138 '<|reserved_special_token_130|>' is not marked as EOG
load: control token: 128136 '<|reserved_special_token_128|>' is not marked as EOG
load: control token: 128135 '<|reserved_special_token_127|>' is not marked as EOG
load: control token: 128134 '<|reserved_special_token_126|>' is not marked as EOG
load: control token: 128133 '<|reserved_special_token_125|>' is not marked as EOG
load: control token: 128131 '<|reserved_special_token_123|>' is not marked as EOG
load: control token: 128128 '<|reserved_special_token_120|>' is not marked as EOG
load: control token: 128124 '<|reserved_special_token_116|>' is not marked as EOG
load: control to

load: control token: 128042 '<|reserved_special_token_34|>' is not marked as EOG
load: control token: 128139 '<|reserved_special_token_131|>' is not marked as EOG
load: control token: 128173 '<|reserved_special_token_165|>' is not marked as EOG
load: control token: 128239 '<|reserved_special_token_231|>' is not marked as EOG
load: control token: 128157 '<|reserved_special_token_149|>' is not marked as EOG
load: control token: 128052 '<|reserved_special_token_44|>' is not marked as EOG
load: control token: 128026 '<|reserved_special_token_18|>' is not marked as EOG
load: control token: 128003 '<|reserved_special_token_1|>' is not marked as EOG
load: control token: 128019 '<|reserved_special_token_11|>' is not marked as EOG
load: control token: 128116 '<|reserved_special_token_108|>' is not marked as EOG
load: control token: 128161 '<|reserved_special_token_153|>' is not marked as EOG
load: control token: 128226 '<|reserved_special_token_218|>' is not marked as EOG
load: control token: 1

load: control token: 128120 '<|reserved_special_token_112|>' is not marked as EOG
load: control token: 128014 '<|reserved_special_token_6|>' is not marked as EOG
load: control token: 128025 '<|reserved_special_token_17|>' is not marked as EOG
load: control token: 128126 '<|reserved_special_token_118|>' is not marked as EOG
load: printing all EOG tokens:
load:   - 128001 ('<|end_of_text|>')
load:   - 128008 ('<|eom_id|>')
load:   - 128009 ('<|eot_id|>')
load: special tokens cache size = 256
load: token to piece cache size = 0.7999 MB
print_info: arch                  = llama
print_info: vocab_only            = 0
print_info: no_alloc              = 0
print_info: n_ctx_train           = 131072
print_info: n_embd_inp            = 2048
print_info: n_embd                = 2048
print_info: n_embd_out            = 2048
print_info: n_layer               = 16
print_info: n_layer_all           = 16
print_info: n_head                = 32
print_info: n_head_kv             = 8
print_info: n_rot     

create_tensor: loading tensor blk.8.ffn_norm.weight
create_tensor: loading tensor blk.8.ffn_gate.weight
create_tensor: loading tensor blk.8.ffn_down.weight
create_tensor: loading tensor blk.8.ffn_up.weight
create_tensor: loading tensor blk.9.attn_norm.weight
create_tensor: loading tensor blk.9.attn_q.weight
create_tensor: loading tensor blk.9.attn_k.weight
create_tensor: loading tensor blk.9.attn_v.weight
create_tensor: loading tensor blk.9.attn_output.weight
create_tensor: loading tensor blk.9.ffn_norm.weight
create_tensor: loading tensor blk.9.ffn_gate.weight
create_tensor: loading tensor blk.9.ffn_down.weight
create_tensor: loading tensor blk.9.ffn_up.weight
create_tensor: loading tensor blk.10.attn_norm.weight
create_tensor: loading tensor blk.10.attn_q.weight
create_tensor: loading tensor blk.10.attn_k.weight
create_tensor: loading tensor blk.10.attn_v.weight
create_tensor: loading tensor blk.10.attn_output.weight
create_tensor: loading tensor blk.10.ffn_norm.weight
create_tensor:

Using gguf chat template: {{- bos_token }}
{%- if custom_tools is defined %}
    {%- set tools = custom_tools %}
{%- endif %}
{%- if not tools_in_user_message is defined %}
    {%- set tools_in_user_message = true %}
{%- endif %}
{%- if not date_string is defined %}
    {%- if strftime_now is defined %}
        {%- set date_string = strftime_now("%d %b %Y") %}
    {%- else %}
        {%- set date_string = "26 Jul 2024" %}
    {%- endif %}
{%- endif %}
{%- if not tools is defined %}
    {%- set tools = none %}
{%- endif %}

{#- This block extracts the system message, so we can slot it into the right place. #}
{%- if messages[0]['role'] == 'system' %}
    {%- set system_message = messages[0]['content']|trim %}
    {%- set messages = messages[1:] %}
{%- else %}
    {%- set system_message = "" %}
{%- endif %}

{#- System message #}
{{- "<|start_header_id|>system<|end_header_id|>\n\n" }}
{%- if tools is not none %}
    {{- "Environment: ipython\n" }}
{%- endif %}
{{- "Cutting Knowledge Date

In [28]:
# Definir la entrada
#prompt = "Act as an astronomer. Name the 8 planets in our Solar system, starting from the closest to the Sun. Output only an ordered list of planet names."
prompt = "Actua como un astronomo. nombra los 8 planetas de nuestro sistema solar, comenzando por el mas proximo al Sol. Responde unicamente con una lista ordenada de los planetas."
# Inferencia con .invoke()
response = llm.invoke(prompt,temperature=0.0)

# Extraer el texto del resultado
# Si es un objeto de LangChain (ChatModel o LLM), .content obtiene la respuesta en texto
final_result = response.content if hasattr(response, "content") else str(response)

print(final_result)

Llama.generate: 2 prefix-match hit, remaining 43 prompt tokens to eval
llama_perf_context_print:        load time =    4296.34 ms
llama_perf_context_print: prompt eval time =   10769.64 ms /    43 tokens (  250.46 ms per token,     3.99 tokens per second)
llama_perf_context_print:        eval time =   21197.57 ms /    62 runs   (  341.90 ms per token,     2.92 tokens per second)
llama_perf_context_print:       total time =   32177.27 ms /   105 tokens
llama_perf_context_print:    graphs reused =         65


 

Los 8 planetas de nuestro sistema solar, comenzando por el mas proximo al Sol:

1. Mercurio
2. Venus
3. Tierra
4. Marte
5. Júpiter
6. Saturno
7. Urano
8. Neptuno


## Setup Fase Retrieval

In [31]:
from src.rag_utils import VectorStoreManager

# Instanciar el gestor de la base de datos
PERSIST_DIRECTORY="../db_chroma"
manager = VectorStoreManager(persist_directory=PERSIST_DIRECTORY, n_ctx=512)

# Procesar un archivo markdown dentro de /knowledgeBase
splits = manager.process_markdown("../data/processed/Algoritmia.md")

# Crear y guardar la base de datos vectorial
vector_db = manager.create_and_persist_db(splits)

Persistiendo 19 fragmentos en '../db_chroma'...
Base de datos vectorial creada con éxito.


## Inicilizacion de base de datos

In [9]:
#Codificacion de los chunks y Creacion de la base de datos
if len(os.listdir(PERSIST_DIRECTORY))==0:
    vector_db = EmbeddDocsAndPersist(chunked_splits,embedding_encoder,PERSIST_DIRECTORY)
else:
    vector_db = load_persisted_db(embedding_encoder,PERSIST_DIRECTORY)

Creando y persistiendo la base de datos de vectores...


init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings required but some input tokens were not marked as outputs -> overriding
init: embeddings requ

Base de datos creada y guardada.


## Testing Fase Retrieval

### Armado de dataset de validacion

In [34]:
##TESTING
prompt=f"Resume el siguiente texto en 3 ideas clave. TEXTO: {chunked_splits[0].page_content}"
max_tokens = 100
temperature = 1
top_p = 0.01
echo = False
stop = ["\n\n\n"]

model_output = llm(
      prompt,
       max_tokens=max_tokens,
       temperature=temperature,
       top_p=top_p,
       echo=echo,
       #stop=stop,
) # Generate a completion, can also call create_completion
final_result = model_output["choices"][0]["text"].strip()
print(model_output["choices"][0]["text"])

Llama.generate: 1 prefix-match hit, remaining 259 prompt tokens to eval




Este texto se centra en la **transformación lineal** y se centra en la **autovalor**.
**Ideas clave:**

*   **Definición:** El texto describe la definición de un vector como una dirección invariante de la transformación lineal.
*   **Características:** El texto describe las características principales del vector como la dirección invariante.
*   **Ejemplos:** El texto proporciona ejemplos concretos para ilustrar la definición y las características.
*   **


In [13]:
##TESTING
prompt=f"Retrieve the chunkID for this question {'¿Cuál es la condición de convergencia para el método de la potencia?'} from this text {chunked_splits[0:6]}"
max_tokens = 50
temperature = 0
top_p = 0.1
echo = False
stop = ["\n\n"]

model_output = llm(
      prompt,
       max_tokens=max_tokens,
       temperature=temperature,
       top_p=top_p,
       echo=echo,
       stop=stop,
) # Generate a completion, can also call create_completion
final_result = model_output["choices"][0]["text"].strip()
print(model_output["choices"][0]["text"])

Llama.generate: 1 prefix-match hit, remaining 2098 prompt tokens to eval


 Document(metadata={'Header 1': '2 MÉTODO DE LA POTENCIA INVERSA', 'doc_id': 'chunk_6'}, page_content='Paso 2: **Lectura de datos**: Leer la matriz `A


In [13]:
evaluation_dataset_AIgenerated = [
    {
        "question": "¿Qué es un autovector y un autovalor?",
        "ground_truth_doc_id": "chunk_0"
    },
    {
        "question": "¿Qué es el polinomio característico y cómo se relaciona con los autovalores?",
        "ground_truth_doc_id": "chunk_0"
    },
    {
        "question": "Menciona las tres categorías de métodos para la determinación de valores y vectores característicos.",
        "ground_truth_doc_id": "chunk_1" 
    },
    {
        "question": "¿En qué consisten los métodos iterativos para encontrar autovalores y cuál es el ejemplo tratado en el curso?",
        "ground_truth_doc_id": "chunk_2"
    },
    {
        "question": "¿Cuál es la condición de convergencia para el método de la potencia directa?",
        "ground_truth_doc_id": "chunk_3"
    },
    {
        "question": "¿Por qué es necesario usar escalamiento en el método de la potencia y cómo se realiza?",
        "ground_truth_doc_id": "chunk_3"
    },
    {
        "question": "¿Cuál es el pseudocodigo del método de la potencia?",
        "ground_truth_doc_id": "chunk_4"
    },
    {
        "question": "¿Cómo funciona el método de la potencia inversa y a qué autovalor de la matriz original converge?",
        "ground_truth_doc_id": "chunk_5"
    },
    {
        "question": "¿Qué es la deflación y para qué se utiliza en el cálculo de autovectores?",
        "ground_truth_doc_id": "chunk_6"
    },
    {
        "question": "¿Qué es la cuadratura y el error de truncamiento en la integración numérica?",
        "ground_truth_doc_id": "chunk_9"
    },
    {
        "question": "¿Cuál es la diferencia fundamental entre la cuadratura de Newton-Cotes y la de Gauss-Legendre?",
        "ground_truth_doc_id": "chunk_11" # This chunk explicitly contrasts with Newton-Cotes described in chunk_10
    },
    {
        "question": "¿Qué es la regla de los trapecios y cuál es su orden de exactitud?",
        "ground_truth_doc_id": "chunk_13"
    },
    {
        "question": "¿Cuál es el orden del error en la regla de los trapecios simple?",
        "ground_truth_doc_id": "chunk_15"
    },
    {
        "question": "¿Cómo cambia el orden del error al pasar de la regla de trapecios simple a la múltiple?",
        "ground_truth_doc_id": "chunk_18"
    },
    {
        "question": "¿Cuál es la fórmula de la regla de Simpson simple y cuántos puntos utiliza?",
        "ground_truth_doc_id": "chunk_21"
    },
    {
        "question": "¿Cuál es el orden del error para la regla de Simpson simple?",
        "ground_truth_doc_id": "chunk_23"
    },
    {
        "question": "¿Cuál es la fórmula de la regla de Simpson compuesta y cuál es el orden de su error?",
        "ground_truth_doc_id": "chunk_24"
    },
    {
        "question": "Describe la regla de cuadratura de Gauss de dos puntos, incluyendo los valores de las abscisas y los coeficientes.",
        "ground_truth_doc_id": "chunk_26"
    },
    {
        "question": "¿Qué es la extrapolación de Richardson y cuál es su propósito?",
        "ground_truth_doc_id": "chunk_28"
    },
    {
        "question": "¿En qué consiste la integración de Romberg y qué técnica aplica sucesivamente?",
        "ground_truth_doc_id": "chunk_29"
    }
]

### Testing vectorstore as retriever

In [14]:
def evaluate_vectorstore_as_retriever(eval_dataset, vector_store, k=5):
    """
    Evaluates the performance of a retriever using a given dataset.

    Args:
        eval_dataset (list): A list of dictionaries with "question" and "ground_truth_doc_id".
        vector_store: The ChromaDB vector store instance.
        k (int): The number of top documents to retrieve for evaluation.

    Returns:
        dict: A dictionary containing the calculated metrics.
    """
    hits = 0
    reciprocal_ranks = []
    misses = [] # To store information about failed queries for later analysis

    print(f"Starting evaluation for k={k}...")

    for item in eval_dataset:
        question = item["question"]
        ground_truth_id = item["ground_truth_doc_id"]
        
        # Perform the similarity search
        # The result is a list of tuples: [(Document, score), (Document, score), ...]
        retrieved_docs_with_scores = vector_store.similarity_search_with_score(question, k=k)
        
        # Extract the doc_ids from the metadata of the retrieved documents
        retrieved_ids = [doc.metadata.get('doc_id') for doc, score in retrieved_docs_with_scores]
        
        # Check if the ground truth ID is in the retrieved IDs
        if ground_truth_id in retrieved_ids:
            hits += 1
            # Find the rank (position) of the correct document. Ranks are 1-based.
            rank = retrieved_ids.index(ground_truth_id) + 1
            reciprocal_ranks.append(1 / rank)
        else:
            reciprocal_ranks.append(0)
            misses.append({
                "question": question,
                "expected": ground_truth_id,
                "retrieved": retrieved_ids
            })

    total_questions = len(eval_dataset)
    hit_rate = (hits / total_questions) * 100
    mrr = np.mean(reciprocal_ranks)

    return {
        "hit_rate_at_k": k,
        "hit_rate": f"{hit_rate:.2f}%",
        "mrr": f"{mrr:.4f}",
        "total_questions": total_questions,
        "hits": hits,
        "misses_count": len(misses),
        "misses": misses
    }

In [15]:
evaluate_vectorstore_as_retriever(evaluation_dataset_AIgenerated, vectorStore, k=5)

Starting evaluation for k=5...


{'hit_rate_at_k': 5,
 'hit_rate': '85.00%',
 'mrr': '0.6917',
 'total_questions': 20,
 'hits': 17,
 'misses_count': 3,
 'misses': [{'question': '¿Cuál es la condición de convergencia para el método de la potencia directa?',
   'expected': 'chunk_3',
   'retrieved': ['chunk_5', 'chunk_17', 'chunk_27', 'chunk_26', 'chunk_13']},
  {'question': '¿Cuál es el pseudocodigo del método de la potencia?',
   'expected': 'chunk_4',
   'retrieved': ['chunk_19', 'chunk_30', 'chunk_5', 'chunk_12', 'chunk_17']},
  {'question': '¿Qué es la deflación y para qué se utiliza en el cálculo de autovectores?',
   'expected': 'chunk_6',
   'retrieved': ['chunk_3', 'chunk_5', 'chunk_0', 'chunk_2', 'chunk_4']}]}

In [16]:
evaluate_vectorstore_as_retriever(evaluation_dataset_AIgenerated, vectorStore, k=3)

Starting evaluation for k=3...


{'hit_rate_at_k': 3,
 'hit_rate': '75.00%',
 'mrr': '0.6667',
 'total_questions': 20,
 'hits': 15,
 'misses_count': 5,
 'misses': [{'question': '¿Cuál es la condición de convergencia para el método de la potencia directa?',
   'expected': 'chunk_3',
   'retrieved': ['chunk_5', 'chunk_17', 'chunk_27']},
  {'question': '¿Cuál es el pseudocodigo del método de la potencia?',
   'expected': 'chunk_4',
   'retrieved': ['chunk_19', 'chunk_30', 'chunk_5']},
  {'question': '¿Qué es la deflación y para qué se utiliza en el cálculo de autovectores?',
   'expected': 'chunk_6',
   'retrieved': ['chunk_3', 'chunk_5', 'chunk_0']},
  {'question': '¿Cuál es la diferencia fundamental entre la cuadratura de Newton-Cotes y la de Gauss-Legendre?',
   'expected': 'chunk_11',
   'retrieved': ['chunk_21', 'chunk_25', 'chunk_14']},
  {'question': 'Describe la regla de cuadratura de Gauss de dos puntos, incluyendo los valores de las abscisas y los coeficientes.',
   'expected': 'chunk_26',
   'retrieved': ['chu

### Testing de Re-ranker

In [23]:
# Creamos el retriever base. Este será "envuelto" por el compresor.
# Le damos un 'k' más alto porque esperamos que el filtro descarte algunos resultados.
base_retriever = vectorStore.as_retriever(
    search_type="similarity_score_threshold", 
    search_kwargs={"score_threshold": 0.1,"k": 10}
    #search_type="similarity", # 'similarity' with a 'k' is often more reliable
    #search_kwargs={"k": 10} # Retrieve more documents to give the reranker more to work with
)
# ¡IMPORTANTE! Usamos la MISMA instancia 'embedding_encoder' que para la DB.
redundant_filter = EmbeddingsRedundantFilter(embeddings=embedding_encoder)
reranker = CrossEncoderReranker(model=retrieval_reranker, top_n=3)
pipeline_compressor = DocumentCompressorPipeline(
    transformers=[redundant_filter, reranker]
)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor, 
    base_retriever=base_retriever
)

In [30]:
def evaluate_retriever(
    retriever: Any,
    eval_dataset: List[Dict[str, str]],
    retriever_name: str
) -> Dict[str, Any]:
    """
    Evaluates the performance of any retriever with a .invoke() method.

    Args:
        retriever (Any): A retriever object that has an .invoke(query) method
                         which returns a list of Document objects.
        eval_dataset (list): A list of dictionaries with "question" and 
                             "ground_truth_doc_id".
        retriever_name (str): A descriptive name for the retriever being tested
                              (e.g., "Base Retriever k=5").

    Returns:
        dict: A dictionary containing the calculated metrics and a list of misses.
    """
    hits = 0
    reciprocal_ranks = []
    misses = []  # To store information about failed queries for analysis

    print(f"--- Starting evaluation for: {retriever_name} ---")

    for item in eval_dataset:
        question = item["question"]
        ground_truth_id = item["ground_truth_doc_id"]
        
        # 1. Use the generic .invoke() method
        retrieved_docs = retriever.invoke(question)
        
        # 2. Extract doc_ids from the list of Document objects
        retrieved_ids = [doc.metadata.get('doc_id') for doc in retrieved_docs]
        
        # 3. Perform the evaluation logic (this part remains the same)
        if ground_truth_id in retrieved_ids:
            hits += 1
            # Ranks are 1-based
            rank = retrieved_ids.index(ground_truth_id) + 1
            reciprocal_ranks.append(1 / rank)
        else:
            reciprocal_ranks.append(0)
            misses.append({
                "question": question,
                "expected": ground_truth_id,
                "retrieved": retrieved_ids
            })

    total_questions = len(eval_dataset)
    hit_rate = (hits / total_questions) * 100
    mrr = np.mean(reciprocal_ranks) if reciprocal_ranks else 0

    print(f"Evaluation finished for: {retriever_name}\n")

    return {
        "retriever_name": retriever_name,
        "hit_rate": f"{hit_rate:.2f}%",
        "mrr": f"{mrr:.4f}",
        "total_questions": total_questions,
        "hits": hits,
        "misses_count": len(misses),
        "misses": misses
    }

def print_results(results: Dict[str, Any]):
    """Helper function to print evaluation results in a readable format."""
    print(f"--- Retrieval Evaluation Results for: {results['retriever_name']} ---")
    print(f"Hit Rate: {results['hit_rate']}")
    print(f"Mean Reciprocal Rank (MRR): {results['mrr']}")
    print(f"Correctly Retrieved (Hits): {results['hits']} / {results['total_questions']}")
    print("-" * 50)

    if results['misses_count'] > 0:
        print(f"\nAnalysis of {results['misses_count']} Misses:")
        # Print details for the first 3 misses for brevity
        for i, miss in enumerate(results['misses'][:3]):
            print(f"\nMiss {i+1}:")
            print(f"  Question: '{miss['question']}'")
            print(f"  Expected Doc ID: {miss['expected']}")
            print(f"  Retrieved IDs:   {miss['retrieved']}")
        if results['misses_count'] > 3:
            print("\n(And more...)")
    print("\n")

In [ ]:
# Evaluate the base retriever
base_retriever_results = evaluate_retriever(
    retriever=base_retriever,
    eval_dataset=evaluation_dataset_AIgenerated,
    retriever_name="Base Retriever"
)
# Evaluate the compression retriever
compression_retriever_results = evaluate_retriever(
    retriever=compression_retriever,
    eval_dataset=evaluation_dataset_AIgenerated,
    retriever_name="Compression Retriever"
)

# --- Reporting Phase ---
print_results(base_retriever_results)
print_results(compression_retriever_results)

--- Starting evaluation for: Base Retriever ---
Evaluation finished for: Base Retriever

--- Starting evaluation for: Compression Retriever ---


In [36]:
print_results(base_retriever_results)

--- Retrieval Evaluation Results for: Base Retriever ---
Hit Rate: 95.00%
Mean Reciprocal Rank (MRR): 0.7083
Correctly Retrieved (Hits): 19 / 20
--------------------------------------------------

Analysis of 1 Misses:

Miss 1:
  Question: '¿Cuál es el pseudocodigo del método de la potencia?'
  Expected Doc ID: chunk_4
  Retrieved IDs:   ['chunk_19', 'chunk_30', 'chunk_5', 'chunk_12', 'chunk_17', 'chunk_21', 'chunk_2', 'chunk_14', 'chunk_28', 'chunk_27']




In [21]:
def DefineGemmaPrompt():
    """
    Defines the prompt template for Gemma 3 models via Google API.
    
    Since Gemma 3 does not support system prompts through this API, this function
    combines the system instructions and the user query into a single human/user
    message template.
    """
    system_instructions = """Eres un experto en pedagogía para estudiantes universitarios de la generación Z y profesor de la cátedra de Métodos Numéricos en la Facultad de Ingeniería. Tu objetivo es guiar al usuario a lograr una comprensión más profunda sobre su pregunta.

Recibirás una PREGUNTA y un CONTEXTO de las notas de clase. Sigue estas reglas estrictamente:
1. Responde a la PREGUNTA utilizando ÚNICAMENTE el CONTEXTO proporcionado. No uses información de otras fuentes. Si no hay CONTEXTO, indica que la respuesta no ha sido encontrada.
2. Formatea siempre tus respuestas utilizando Markdown para mejorar la legibilidad.
3. Después de tu explicación, incluye una sección de TAREAS ACCIONABLES o PREGUNTAS DE REFLEXIÓN para que el estudiante aplique o profundice su conocimiento.
4. Responde siempre en español. Sé útil y claro."""

    # Combine the instructions and the dynamic parts into a single template string.
    # The model will treat the entire block as the user's input.
    full_prompt_string = (
        f"{system_instructions}\n\n"
        "--- \n\n"  # Using a separator can sometimes help the model distinguish instructions from data.
        "CONTEXTO:\n{context}\n\n"
        "PREGUNTA:\n{question}"
    )

    # Create the template from a single string. LangChain will treat this
    # as a single "human" message by default in many chains.
    prompt_template = ChatPromptTemplate.from_template(full_prompt_string)
    
    return prompt_template

def ProcessInput(question,retriever,llm):
    #Data pipeline: user query->retrieve chunks->join them->inject in prompt-> get LLM response
    prompt = DefineGemmaPrompt()
    rag_chain = (
        {"context": retriever | join_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return rag_chain.invoke(question)

In [19]:
#llm=LoadOllamaLLM()
#llm=LoadGoogleLLM()
llm=LoadLLMFromllamaCPP()

In [24]:
#Uso de herramienta
query="Que pasa si ingreso un autovector como vector inicial en el metodo de la potencia?"

response=ProcessInput(query,base_retriever,llm)
print(response)

De acuerdo al contexto proporcionado, si ingresas un autovector como vector inicial en el método de la potencia, el proceso se simplifica significativamente. 

El método de la potencia, como se explica, converge al autovector dominante (el asociado al autovalor de mayor valor absoluto) a través de sucesivas premultiplicaciones de un vector inicial por la matriz `A`.  Si el vector inicial `x` ya es un autovector `v_1` asociado al autovalor dominante `λ_1`, entonces:

`A x = A v_1 = λ_1 v_1 = λ_1 x`

Cada iteración del método simplemente multiplicará el vector `x` por el autovalor `λ_1`.  El vector permanecerá en la dirección del autovector `v_1` y el cociente `x_{k+1}(j) / x_k(j)` convergerá inmediatamente a `λ_1` sin necesidad de múltiples iteraciones.  El escalamiento (normalización) seguirá siendo necesario para evitar problemas de overflow o underflow, pero la convergencia será mucho más rápida y directa.

---

**TAREAS ACCIONABLES / PREGUNTAS DE REFLEXIÓN:**

1.  **Considera una ma